# Amazon Bedrock Knowledge Base — EU Gambling & Betting Compliance

This notebook creates a Bedrock Knowledge Base hydrated with EU legal/compliance documents
for the Betting, Gaming, and Gambling industry.

**Documents source:** `s3://agentic-ai-demo-kb-data` (pre-uploaded)

**What this notebook does:**
1. Creates an OpenSearch Serverless vector collection
2. Creates a Bedrock Knowledge Base
3. Creates a data source pointing to the existing S3 bucket
4. Ingests the documents
5. Queries the KB (raw retrieval + AI-generated answers)
6. Outputs the KB ID/ARN for use in other notebooks
7. Cleanup

## Step 0 — Install Dependencies

In [1]:
!pip install -q boto3 opensearch-py

## Step 1 — Configuration

In [2]:
import boto3
import json
import time

REGION = "us-east-1"
ACCOUNT_ID = "869795389844"
ROLE_NAME = "AgentCoreDemoRole"
role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"

# Your documents are already in this bucket
S3_BUCKET_NAME = "agentic-ai-demo-kb-data"

KB_NAME = "eu-gambling-compliance-kb"
KB_DESCRIPTION = "Knowledge Base with EU legal/compliance documents for Betting, Gaming, and Gambling"
EMBEDDING_MODEL_ARN = f"arn:aws:bedrock:{REGION}::foundation-model/amazon.titan-embed-text-v2:0"
GENERATION_MODEL = f"arn:aws:bedrock:{REGION}::foundation-model/amazon.nova-pro-v1:0"

# Clients
s3 = boto3.client("s3", region_name=REGION)
iam = boto3.client("iam")
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)
aoss = boto3.client("opensearchserverless", region_name=REGION)

# Get caller identity for access policies
identity = boto3.client("sts", region_name=REGION).get_caller_identity()

print(f"Account  : {ACCOUNT_ID}")
print(f"Region   : {REGION}")
print(f"Role     : {role_arn}")
print(f"S3 Bucket: {S3_BUCKET_NAME}")
print(f"Embed    : {EMBEDDING_MODEL_ARN}")
print(f"Generate : {GENERATION_MODEL}")

Account  : 869795389844
Region   : us-east-1
Role     : arn:aws:iam::869795389844:role/AgentCoreDemoRole
S3 Bucket: agentic-ai-demo-kb-data
Embed    : arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-embed-text-v2:0
Generate : arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-pro-v1:0


## Step 4 — Create OpenSearch Serverless Collection & Bedrock Knowledge Base

Creates the vector collection, index, and Knowledge Base. Takes ~5 minutes.

In [5]:
COLLECTION_NAME = "eu-gambling-kb-vectors"
INDEX_NAME = "bedrock-kb-index"

# 1. Encryption policy
try:
    aoss.create_security_policy(
        name=f"{COLLECTION_NAME}-enc", type="encryption",
        policy=json.dumps({"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}], "AWSOwnedKey": True}))
    print("Created encryption policy")
except aoss.exceptions.ConflictException:
    print("Encryption policy already exists")

# 2. Network policy
try:
    aoss.create_security_policy(
        name=f"{COLLECTION_NAME}-net", type="network",
        policy=json.dumps([{"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}, {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]}], "AllowFromPublic": True}]))
    print("Created network policy")
except aoss.exceptions.ConflictException:
    print("Network policy already exists")

# 3. Data access policy
try:
    aoss.create_access_policy(
        name=f"{COLLECTION_NAME}-access", type="data",
        policy=json.dumps([{"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"], "Permission": ["aoss:CreateCollectionItems", "aoss:UpdateCollectionItems", "aoss:DescribeCollectionItems"]}, {"ResourceType": "index", "Resource": [f"index/{COLLECTION_NAME}/*"], "Permission": ["aoss:CreateIndex", "aoss:UpdateIndex", "aoss:DescribeIndex", "aoss:ReadDocument", "aoss:WriteDocument"]}], "Principal": [role_arn, identity["Arn"]], "Description": "KB access"}]))
    print("Created data access policy")
except aoss.exceptions.ConflictException:
    print("Data access policy already exists")

# 4. Create collection
try:
    coll_resp = aoss.create_collection(name=COLLECTION_NAME, type="VECTORSEARCH", description="Vector store for EU gambling KB")
    collection_id = coll_resp["createCollectionDetail"]["id"]
    print(f"Creating collection: {collection_id}")
except aoss.exceptions.ConflictException:
    colls = aoss.batch_get_collection(names=[COLLECTION_NAME])
    collection_id = colls["collectionDetails"][0]["id"]
    print(f"Collection already exists: {collection_id}")

# Wait for ACTIVE
print("Waiting for collection to become ACTIVE (2-5 min)...")
for _ in range(30):
    colls = aoss.batch_get_collection(ids=[collection_id])
    coll_status = colls["collectionDetails"][0]["status"]
    if coll_status == "ACTIVE":
        break
    print(f"  Collection status: {coll_status}")
    time.sleep(15)

collection_arn = colls["collectionDetails"][0]["arn"]
collection_endpoint = colls["collectionDetails"][0]["collectionEndpoint"]
print(f"Collection ACTIVE: {collection_arn}")

Created encryption policy
Created network policy
Created data access policy
Creating collection: jx609ajaqeftd1u5h6d2
Waiting for collection to become ACTIVE (2-5 min)...
  Collection status: CREATING
  Collection status: CREATING
  Collection status: CREATING
  Collection status: CREATING
Collection ACTIVE: arn:aws:aoss:us-east-1:869795389844:collection/jx609ajaqeftd1u5h6d2


In [6]:
# 5. Create vector index
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
credentials = boto3.Session().get_credentials()
auth = AWSV4SignerAuth(credentials, REGION, "aoss")
host = collection_endpoint.replace("https://", "")
oss_client = OpenSearch(hosts=[{"host": host, "port": 443}], http_auth=auth, use_ssl=True, verify_certs=True, connection_class=RequestsHttpConnection, timeout=300)

index_body = {
    "settings": {"index.knn": True, "number_of_shards": 2, "number_of_replicas": 0},
    "mappings": {"properties": {
        "bedrock-knowledge-base-default-vector": {"type": "knn_vector", "dimension": 1024, "method": {"engine": "faiss", "name": "hnsw", "parameters": {}, "space_type": "l2"}},
        "AMAZON_BEDROCK_TEXT_CHUNK": {"type": "text"},
        "AMAZON_BEDROCK_METADATA": {"type": "text"}
    }}
}
try:
    oss_client.indices.create(index=INDEX_NAME, body=index_body)
    print(f"Created vector index: {INDEX_NAME}")
except Exception as e:
    if "already exists" in str(e).lower() or "resource_already_exists" in str(e).lower():
        print(f"Index already exists: {INDEX_NAME}")
    else:
        raise e

# Wait for index propagation
print("Waiting 60s for index to propagate...")
time.sleep(60)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Created vector index: bedrock-kb-index
Waiting 60s for index to propagate...


In [7]:
# 6. Create Knowledge Base
kb_response = bedrock_agent.create_knowledge_base(
    name=KB_NAME, description=KB_DESCRIPTION, roleArn=role_arn,
    knowledgeBaseConfiguration={"type": "VECTOR", "vectorKnowledgeBaseConfiguration": {
        "embeddingModelArn": EMBEDDING_MODEL_ARN,
        "embeddingModelConfiguration": {"bedrockEmbeddingModelConfiguration": {"dimensions": 1024}}}},
    storageConfiguration={"type": "OPENSEARCH_SERVERLESS", "opensearchServerlessConfiguration": {
        "collectionArn": collection_arn, "vectorIndexName": INDEX_NAME,
        "fieldMapping": {"vectorField": "bedrock-knowledge-base-default-vector",
                         "textField": "AMAZON_BEDROCK_TEXT_CHUNK", "metadataField": "AMAZON_BEDROCK_METADATA"}}})

kb_id = kb_response["knowledgeBase"]["knowledgeBaseId"]
kb_arn = f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/{kb_id}"
print(f"\nKnowledge Base ID : {kb_id}")
print(f"Knowledge Base ARN: {kb_arn}")


Knowledge Base ID : MCJNMES8UI
Knowledge Base ARN: arn:aws:bedrock:us-east-1:869795389844:knowledge-base/MCJNMES8UI


## Step 5 — Create Data Source & Ingest Documents

Points the KB at your existing S3 bucket `s3://agentic-ai-demo-kb-data`.

In [8]:
# Wait for KB to become ACTIVE
for _ in range(30):
    status = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"]
    if status == "ACTIVE":
        break
    print(f"  KB status: {status}")
    time.sleep(10)
print(f"KB is {status}")

# Create data source pointing to existing S3 bucket (no inclusionPrefixes — ingest everything)
ds_response = bedrock_agent.create_data_source(
    knowledgeBaseId=kb_id, name="eu-gambling-compliance-s3",
    description="EU gambling/betting compliance documents",
    dataSourceConfiguration={"type": "S3", "s3Configuration": {
        "bucketArn": f"arn:aws:s3:::{S3_BUCKET_NAME}"}},
    vectorIngestionConfiguration={"chunkingConfiguration": {
        "chunkingStrategy": "FIXED_SIZE",
        "fixedSizeChunkingConfiguration": {"maxTokens": 512, "overlapPercentage": 20}}})
ds_id = ds_response["dataSource"]["dataSourceId"]
print(f"Data source ID: {ds_id}")

# Trigger ingestion
sync = bedrock_agent.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
job_id = sync["ingestionJob"]["ingestionJobId"]
print(f"\nIngestion job: {job_id}")
while True:
    job = bedrock_agent.get_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id)
    st = job["ingestionJob"]["status"]
    stats = job["ingestionJob"].get("statistics", {})
    print(f"  {st} | Scanned: {stats.get('numberOfDocumentsScanned',0)} | Indexed: {stats.get('numberOfNewDocumentsIndexed',0)} | Failed: {stats.get('numberOfDocumentsFailed',0)}")
    if st in ("COMPLETE", "FAILED", "STOPPED"):
        break
    time.sleep(10)
print(f"\nIngestion {st}!")

KB is ACTIVE
Data source ID: 8CYYD0LI2U

Ingestion job: 9CKWH7AS0T
  STARTING | Scanned: 0 | Indexed: 0 | Failed: 0
  IN_PROGRESS | Scanned: 11 | Indexed: 0 | Failed: 0
  IN_PROGRESS | Scanned: 11 | Indexed: 1 | Failed: 0
  IN_PROGRESS | Scanned: 11 | Indexed: 1 | Failed: 0
  COMPLETE | Scanned: 11 | Indexed: 11 | Failed: 0

Ingestion COMPLETE!


## Step 6 — Query the Knowledge Base

Two modes:
- **Retrieve** — raw document chunks with relevance scores
- **RetrieveAndGenerate** — AI-generated answer grounded in your documents

In [9]:
bedrock_rt = boto3.client("bedrock-agent-runtime", region_name=REGION)

def retrieve(query, n=5):
    resp = bedrock_rt.retrieve(knowledgeBaseId=kb_id, retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": n}})
    return [{"text": r["content"]["text"], "score": r.get("score",0),
             "source": r.get("location",{}).get("s3Location",{}).get("uri","")}
            for r in resp.get("retrievalResults",[])]

def ask(question):
    resp = bedrock_rt.retrieve_and_generate(
        input={"text": question},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": {
                "knowledgeBaseId": kb_id, "modelArn": GENERATION_MODEL,
                "retrievalConfiguration": {"vectorSearchConfiguration": {"numberOfResults": 5}},
                "generationConfiguration": {"inferenceConfig": {"textInferenceConfig": {"maxTokens": 1024, "temperature": 0.0}}}
            }
        })
    sources = list({ref.get("location",{}).get("s3Location",{}).get("uri","")
                   for c in resp.get("citations",[]) for ref in c.get("retrievedReferences",[])})
    return resp["output"]["text"], sources

print(f"Using model: {GENERATION_MODEL}")
print("Query functions ready: retrieve() and ask()")

Using model: arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-pro-v1:0
Query functions ready: retrieve() and ask()


### Test: Raw Retrieval

In [10]:
chunks = retrieve("What are the AML obligations for online gambling operators in the EU?")
for i, c in enumerate(chunks, 1):
    print(f"--- Result {i} (score: {c['score']:.4f}) ---")
    print(f"Source: {c['source']}")
    print(c['text'][:400])
    print()

--- Result 1 (score: 0.6339) ---
Source: s3://agentic-ai-demo-kb-data/09_AML_Sanctions_Compliance_iGaming.pdf
09 AML Sanctions Compliance iGaming     AML and Sanctions Screening in iGaming dilisense Open main menu Products Pricing Data Sources Use Cases Developers Company Sign in Register     AML and Sanctions compliance in iGaming Guidance through the essential stages of AML compliance and sanctions screening in the iGaming space.     Published May 18, 2025     TL;DR iGaming operators in Europe must adhe

--- Result 2 (score: 0.5812) ---
Source: s3://agentic-ai-demo-kb-data/11_EU_Commission_Recommendation_2014_478_Online_Gambling.pdf
INFORMATION REQUIREMENTS     4. Gambling operators should make easily accessible to players, before they register, the following information: (a) the identity of the operator, including its trade name, registered address and contact details; (b) the conditions for participation in gambling, including age restrictions; (c) the rules of the game, including 

### Test: AI-Generated Answers

In [11]:
questions = [
    "What are the key AML obligations for online gambling operators under EU law?",
    "How does GDPR apply to betting operators? What data can they collect?",
    "What customer due diligence is required for gambling under EU AML Regulation 2024/1624?",
    "What are the responsible gambling requirements from the EU Commission Recommendation?",
    "Can a player request deletion of their data under GDPR if AML rules require retention?",
    "What penalties can gambling operators face for AML non-compliance?",
]

for q in questions:
    print("=" * 80)
    print(f"Q: {q}")
    a, srcs = ask(q)
    print(f"\nA: {a}")
    if srcs:
        print(f"\nSources: {srcs}")
    print()

Q: What are the key AML obligations for online gambling operators under EU law?

A: Based on the retrieved results, the key AML obligations for online gambling operators under EU law are:

1. **Customer Due Diligence (CDD)**: Operators must conduct CDD during account creation or when deposits reach specific thresholds.

2. **Identity Verification**: Operators are required to verify the identities of their customers.

3. **Enhanced Due Diligence (EDD)**: Operators must conduct EDD for high-risk customers, such as Politically Exposed Persons (PEPs).

4. **Sanctions Screening**: Operators must screen customers against sanctions lists from the EU, UN, and OFAC.

5. **Suspicious Activity Reports (SARs)**: Operators are obligated to file SARs for any suspicious transactions.

6. **Record-Keeping**: Operators must retain transaction and CDD data for at least five years.

7. **AML Compliance Officer**: Operators must appoint an Anti-Money Laundering Reporting Officer (MLRO) to oversee complian

## Step 7 — Knowledge Base Details

Use these values to reference the KB from other notebooks or agents.

In [12]:
print(f"Knowledge Base ID  : {kb_id}")
print(f"Knowledge Base ARN : {kb_arn}")
print(f"S3 Bucket          : {S3_BUCKET_NAME}")
print(f"OpenSearch ARN     : {collection_arn}")
print(f"IAM Role ARN       : {role_arn}")
print(f"Generation Model   : {GENERATION_MODEL}")
print(f"Region             : {REGION}")

Knowledge Base ID  : MCJNMES8UI
Knowledge Base ARN : arn:aws:bedrock:us-east-1:869795389844:knowledge-base/MCJNMES8UI
S3 Bucket          : agentic-ai-demo-kb-data
OpenSearch ARN     : arn:aws:aoss:us-east-1:869795389844:collection/jx609ajaqeftd1u5h6d2
IAM Role ARN       : arn:aws:iam::869795389844:role/AgentCoreDemoRole
Generation Model   : arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-pro-v1:0
Region             : us-east-1


## Step 8 — Cleanup (Optional)

**Uncomment the lines below to delete all resources and stop charges.**

Note: This does NOT delete the S3 bucket `agentic-ai-demo-kb-data` since it's shared.

In [ ]:
# UNCOMMENT TO DELETE:
# bedrock_agent.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id); print("Deleted data source")
# bedrock_agent.delete_knowledge_base(knowledgeBaseId=kb_id); print("Deleted KB")
# aoss.delete_collection(id=collection_id); print("Deleted OpenSearch collection")
# aoss.delete_access_policy(name=f"{COLLECTION_NAME}-access", type="data"); print("Deleted access policy")
# aoss.delete_security_policy(name=f"{COLLECTION_NAME}-net", type="network"); print("Deleted network policy")
# aoss.delete_security_policy(name=f"{COLLECTION_NAME}-enc", type="encryption"); print("Deleted encryption policy")
print("Cleanup section ready — uncomment lines above to execute")